# CIFAR10 Neural Networks w/ Resnet Architecture

In [6]:
import os
import torch
import torchvision
import torch.nn as nn
from tqdm import tqdm
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: ", device)

Device:  cuda


## Data Augmentation

In [7]:
train_transform = transforms.Compose([
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    transforms.RandomPerspective(distortion_scale=0.5, p=0.5),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
])

## Data Loading — 80/20 Split

All 60,000 CIFAR10 images (50k train + 10k test) are combined and split 80/20:
- **Train**: 48,000 samples
- **Test**: 12,000 samples

In [13]:
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']


class CIFAR10Combined(Dataset):
    """Wraps a numpy array + label list so we can apply per-split transforms."""

    def __init__(self, data, labels, transform=None):
        self.data      = data       # (N, H, W, C) uint8 numpy
        self.labels    = labels     # list[int]
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = Image.fromarray(self.data[idx])
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(self.labels[idx])
        label = F.one_hot(label, num_classes=10).float()
        return {"img": image, "label": label}


# Download and combine both official splits
print("Loading CIFAR10 dataset...")
raw_train = torchvision.datasets.CIFAR10('.data/', train=True,  download=True)
raw_test  = torchvision.datasets.CIFAR10('.data/', train=False, download=True)

all_data   = np.concatenate([raw_train.data, raw_test.data], axis=0)  # (60000, 32, 32, 3)
all_labels = raw_train.targets + raw_test.targets                      # 60000 ints

# 80/20 split
split = int(0.8 * len(all_data))   # 48000
print(f"Total: {len(all_data)}  |  Train: {split}  |  Test: {len(all_data) - split}")

train_dataset = CIFAR10Combined(all_data[:split],  all_labels[:split],  transform=train_transform)
test_dataset  = CIFAR10Combined(all_data[split:],  all_labels[split:],  transform=test_transform)

# drop_last=True ensures every training batch is divisible by GhostBatchNorm's num_splits=16
batch_size       = 128
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  drop_last=True,  num_workers=0)
test_dataloader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, drop_last=False, num_workers=0)
print(f"Train batches: {len(train_dataloader)}  |  Test batches: {len(test_dataloader)}")

Loading CIFAR10 dataset...


HTTPError: HTTP Error 503: Service Unavailable

## Model Definition

In [9]:
import torch.nn.functional as F


class GhostBatchNorm(nn.BatchNorm2d):
    def __init__(self, num_features, num_splits, **kw):
        super().__init__(num_features, **kw)
        running_mean = torch.zeros(num_features * num_splits)
        running_var  = torch.ones(num_features * num_splits)
        self.weight.requires_grad = False
        self.num_splits = num_splits
        self.register_buffer("running_mean", running_mean)
        self.register_buffer("running_var",  running_var)

    def train(self, mode=True):
        if (self.training is True) and (mode is False):
            self.running_mean = torch.mean(
                self.running_mean.view(self.num_splits, self.num_features), dim=0
            ).repeat(self.num_splits)
            self.running_var = torch.mean(
                self.running_var.view(self.num_splits, self.num_features), dim=0
            ).repeat(self.num_splits)
        return super().train(mode)

    def forward(self, input):
        n, c, h, w = input.shape
        if self.training or not self.track_running_stats:
            assert n % self.num_splits == 0, (
                f"Batch size ({n}) must be divisible by num_splits ({self.num_splits})"
            )
            return F.batch_norm(
                input.view(-1, c * self.num_splits, h, w),
                self.running_mean, self.running_var,
                self.weight.repeat(self.num_splits),
                self.bias.repeat(self.num_splits),
                True, self.momentum, self.eps,
            ).view(n, c, h, w)
        else:
            return F.batch_norm(
                input,
                self.running_mean[: self.num_features],
                self.running_var[: self.num_features],
                self.weight, self.bias,
                False, self.momentum, self.eps,
            )


def conv_bn_relu(c_in, c_out, kernel_size=(3, 3), padding=(1, 1)):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, kernel_size=kernel_size, padding=padding, bias=False),
        GhostBatchNorm(c_out, num_splits=16),
        nn.CELU(alpha=0.3),
    )


def conv_pool_norm_act(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, kernel_size=(3, 3), padding=(1, 1), bias=False),
        nn.MaxPool2d(kernel_size=2, stride=2),
        GhostBatchNorm(c_out, num_splits=16),
        nn.CELU(alpha=0.3),
    )


def patch_whitening(data, patch_size=(3, 3)):
    h, w     = patch_size
    c        = data.size(1)
    patches  = data.unfold(2, h, 1).unfold(3, w, 1)
    patches  = patches.transpose(1, 3).reshape(-1, c, h, w).to(torch.float32)
    n, c, h, w = patches.shape
    X        = patches.reshape(n, c * h * w)
    X        = X / (X.size(0) - 1) ** 0.5
    cov      = X.t() @ X
    eigenvalues, eigenvectors = torch.linalg.eigh(cov)
    eigenvalues  = eigenvalues.flip(0)
    eigenvectors = eigenvectors.t().reshape(c * h * w, c, h, w).flip(0)
    return eigenvectors / torch.sqrt(eigenvalues + 1e-2).view(-1, 1, 1, 1)


class ResNetBagOfTricks(nn.Module):
    def __init__(self, first_layer_weights, c_in, c_out, scale_out):
        super().__init__()
        c      = first_layer_weights.size(0)
        conv1  = nn.Conv2d(c_in, c, kernel_size=(3, 3), padding=(1, 1), bias=False)
        conv1.weight.data        = first_layer_weights
        conv1.weight.requires_grad = False
        self.conv1   = conv1
        self.conv2   = conv_bn_relu(c,   64,  kernel_size=(1, 1), padding=0)
        self.conv3   = conv_pool_norm_act(64,  128)
        self.conv4   = conv_bn_relu(128, 128)
        self.conv5   = conv_bn_relu(128, 128)
        self.conv6   = conv_pool_norm_act(128, 256)
        self.conv7   = conv_pool_norm_act(256, 512)
        self.conv8   = conv_bn_relu(512, 512)
        self.conv9   = conv_bn_relu(512, 512)
        self.pool10  = nn.MaxPool2d(kernel_size=4, stride=4)
        self.linear11 = nn.Linear(512, c_out, bias=False)
        self.scale_out = scale_out

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = x + self.conv5(self.conv4(x))
        x = self.conv6(x)
        x = self.conv7(x)
        x = x + self.conv9(self.conv8(x))
        x = self.pool10(x)
        x = x.reshape(x.size(0), x.size(1))
        x = self.linear11(x)
        return self.scale_out * x


# Compute patch-whitening weights from first 10k training images (normalised to [0,1])
_raw_np = train_dataset.data[:10000]                          # (N, H, W, C) uint8
_raw    = torch.tensor(_raw_np / 255.0, dtype=torch.float32).permute(0, 3, 1, 2)
pw_weights = patch_whitening(_raw)

net = ResNetBagOfTricks(pw_weights, c_in=3, c_out=10, scale_out=0.125)
print(net)
print(f"\nTrainable parameters: {sum(p.numel() for p in net.parameters() if p.requires_grad):,}")

ResNetBagOfTricks(
  (conv1): Conv2d(3, 27, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (conv2): Sequential(
    (0): Conv2d(27, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (1): GhostBatchNorm(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): CELU(alpha=0.3)
  )
  (conv3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (2): GhostBatchNorm(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): CELU(alpha=0.3)
  )
  (conv4): Sequential(
    (0): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): GhostBatchNorm(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): CELU(alpha=0.3)
  )
  (conv5): Sequential(
    (0): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): GhostB

## Training

16-epoch training loop with SGD + Nesterov momentum and cosine-annealing LR schedule.
Per-epoch **train loss**, **train accuracy**, **test loss**, and **test accuracy** are printed and stored in `history`.

In [11]:
epochs    = 16
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

net.to(device)

history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
best_acc, best_epoch = 0.0, 0

header = f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Test Loss':>9} | {'Test Acc':>8} | {'LR':>9}"
print(header)
print('-' * len(header))

for epoch in range(1, epochs + 1):

    # ── Training ─────────────────────────────────────────────────────────────
    net.train()
    run_loss, run_correct = 0.0, 0

    pbar = tqdm(train_dataloader, desc=f"[Train {epoch:02d}/{epochs}]", leave=False)
    for batch in pbar:
        imgs   = batch["img"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        out  = net(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()

        run_loss    += loss.item() * imgs.size(0)
        hard_labels  = labels.argmax(dim=1)
        run_correct += out.argmax(dim=1).eq(hard_labels).sum().item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = run_loss    / len(train_dataset)
    train_acc  = 100.0 * run_correct / len(train_dataset)
    scheduler.step()

    # ── Validation (test set) ────────────────────────────────────────────────
    net.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc=f"[Val   {epoch:02d}/{epochs}]", leave=False):
            imgs   = batch["img"].to(device)
            labels = batch["label"].to(device)
            out    = net(imgs)
            loss   = criterion(out, labels)

            val_loss    += loss.item() * imgs.size(0)
            hard_labels  = labels.argmax(dim=1)
            val_correct += out.argmax(dim=1).eq(hard_labels).sum().item()

    val_loss = val_loss    / len(test_dataset)
    val_acc  = 100.0 * val_correct / len(test_dataset)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(val_loss)
    history['test_acc'].append(val_acc)

    cur_lr = optimizer.param_groups[0]['lr']
    print(f"{epoch:>6} | {train_loss:>10.4f} | {train_acc:>8.2f}% | {val_loss:>9.4f} | {val_acc:>7.2f}% | {cur_lr:>9.6f}")

    if val_acc > best_acc:
        best_acc, best_epoch = val_acc, epoch
        torch.save(net.state_dict(), "best_model.pt")

print(f"\nBest Validation Accuracy: {best_acc:.2f}%  (Epoch {best_epoch})")

 Epoch | Train Loss | Train Acc | Test Loss | Test Acc |        LR
------------------------------------------------------------------


[Train 01/16]:   0%|          | 0/375 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Testing

Load the best saved checkpoint and run a final evaluation on the test set.
Prints overall loss/accuracy and a per-class breakdown.

In [ ]:
net.load_state_dict(torch.load("best_model.pt", map_location=device))
net.eval()

all_preds, all_true     = [], []
total_loss, total_correct = 0.0, 0

with torch.no_grad():
    for batch in tqdm(test_dataloader, desc="Final Test Evaluation"):
        imgs   = batch["img"].to(device)
        labels = batch["label"].to(device)
        out    = net(imgs)
        loss   = criterion(out, labels)

        total_loss    += loss.item() * imgs.size(0)
        hard_labels    = labels.argmax(dim=1)
        total_correct += out.argmax(dim=1).eq(hard_labels).sum().item()

        all_preds.extend(out.argmax(dim=1).cpu().numpy())
        all_true.extend(hard_labels.cpu().numpy())

final_loss = total_loss / len(test_dataset)
final_acc  = 100.0 * total_correct / len(test_dataset)

print(f"\n{'='*46}")
print(f"  Final Test Loss    : {final_loss:.4f}")
print(f"  Final Test Accuracy: {final_acc:.2f}%")
print(f"{'='*46}\n")

all_preds = np.array(all_preds)
all_true  = np.array(all_true)

print(f"{'Class':<14} {'Correct':>8} {'Total':>7} {'Accuracy':>10}")
print('-' * 42)
for i, cls in enumerate(CLASSES):
    mask    = all_true == i
    correct = (all_preds[mask] == i).sum()
    total   = mask.sum()
    print(f"{cls:<14} {correct:>8} {total:>7} {100.*correct/total:>9.2f}%")

## Results

Training and test loss / accuracy curves plotted side by side across all 16 epochs.

In [ ]:
epoch_range = range(1, epochs + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Loss ──────────────────────────────────────────────────────────────────────
ax1.plot(epoch_range, history['train_loss'], 'b-o', label='Train Loss', linewidth=2, markersize=5)
ax1.plot(epoch_range, history['test_loss'],  'r-s', label='Test Loss',  linewidth=2, markersize=5)
ax1.set_title('Loss per Epoch', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xticks(epoch_range)

# ── Accuracy ──────────────────────────────────────────────────────────────────
ax2.plot(epoch_range, history['train_acc'], 'b-o', label='Train Accuracy', linewidth=2, markersize=5)
ax2.plot(epoch_range, history['test_acc'],  'r-s', label='Test Accuracy',  linewidth=2, markersize=5)
ax2.set_title('Accuracy per Epoch', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xticks(epoch_range)

plt.suptitle('ResNetBagOfTricks — CIFAR10 (80/20 Split, 16 Epochs)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_results.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved → training_results.png")

## Random Image Classification

Pick **8 random images** from the test set, run the model, and display each image with its predicted and true label.
Titles are **green** when the prediction is correct and **red** when wrong.

In [ ]:
net.load_state_dict(torch.load("best_model.pt", map_location=device))
net.eval()

rng     = np.random.default_rng(seed=99)
indices = rng.choice(len(test_dataset), size=8, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

for ax_idx, img_idx in enumerate(indices):
    raw_img    = test_dataset.data[img_idx]        # (32, 32, 3) uint8 — for display
    true_label = test_dataset.labels[img_idx]

    # Prepare tensor for model inference
    pil_img  = Image.fromarray(raw_img)
    input_t  = test_transform(pil_img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits     = net(input_t)
        probs      = torch.softmax(logits, dim=1)[0]
        pred_label = probs.argmax().item()
        confidence = probs[pred_label].item()

    correct = (pred_label == true_label)
    color   = 'green' if correct else 'red'
    title   = (f"Pred: {CLASSES[pred_label]} ({confidence*100:.1f}%)\n"
               f"True: {CLASSES[true_label]}")

    axes[ax_idx].imshow(raw_img)
    axes[ax_idx].set_title(title, color=color, fontsize=9, fontweight='bold')
    axes[ax_idx].axis('off')

plt.suptitle('CIFAR10 — 8 Random Test Predictions  (green = correct, red = wrong)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('random_predictions.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved → random_predictions.png")